In [ ]:
import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=True)
repo_root = ctx.repo_root

import datajoint as dj


In [ ]:
# Setup
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')

import datajoint as dj
dj.conn()

import pathlib
import pandas as pd
from tqdm.notebook import tqdm

from adamacs.pipeline import subject, session, scan, model

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

USER_INITIALS = 'NK'
START_DATE = '2025-03-01'

# DLC Model - same model used for BOTH eyes
# The search string "eye1_video" is extracted from position [2] after splitting by ';'
DLC_MODEL_NAME = 'NK; General_eye_fullsize-NK-2025-08-25; eye1_video'
DLC_DEINT_MODEL_NAME = 'NK; General_eye_deinterlaced-NK-2025-07-17; eye1_video'

# Camera configuration (from NK.ini)
# camera2 = left eye (uses eye1_video pattern)
# camera3 = right eye (uses eye2_video pattern - eye1 replaced with eye2)
CAMERAS = {
    'left': 'mini2p1_eye_left',   # eye1_video pattern
    'right': 'mini2p1_eye_right'  # eye2_video pattern (model converts eye1->eye2)
}

# Extract search string from model name (matches adamacs_ingest_v2.py logic)
SEARCH_STR_LEFT = DLC_MODEL_NAME.split(';')[2].replace(' ', '')  # eye1_video
SEARCH_STR_RIGHT = SEARCH_STR_LEFT.replace('eye1', 'eye2')       # eye2_video

# Processing options
USE_DYNAMIC_CROPPING = False  # From NK.ini: use_dlc_cropping = False
TASK_MODE = 'trigger'

print(f"User: {USER_INITIALS}")
print(f"Start Date: {START_DATE}")
print(f"Model: {DLC_MODEL_NAME}")
print(f"Deinterlaced Model: {DLC_DEINT_MODEL_NAME}")
print(f"Search patterns: Left='{SEARCH_STR_LEFT}', Right='{SEARCH_STR_RIGHT}'")
print(f"Dynamic Cropping: {USE_DYNAMIC_CROPPING}")

In [ ]:
# Verify model exists
model_query = model.Model & f'model_name = "{DLC_MODEL_NAME}"'
assert model_query, f"Model not found: {DLC_MODEL_NAME}"
print(f"✅ Model found: {DLC_MODEL_NAME}")

In [ ]:
# Get sessions and scans
session_query = (
    session.Session * session.SessionUser * subject.User 
    & f"initials = '{USER_INITIALS}'"
    & f"session_datetime >= '{START_DATE}'"
)

sessions_df = session_query.fetch(format='frame').reset_index()
session_ids = sessions_df['session_id'].tolist()

scans_query = scan.Scan * scan.ScanPath & [f'session_id = "{sid}"' for sid in session_ids]
scans_df = scans_query.fetch(format='frame').reset_index()

print(f"Sessions: {len(sessions_df)}")
print(f"Scans with paths: {len(scans_df)}")

In [ ]:
# Find eye videos in scan directories
# Prioritize 'deinterlaced' videos when available (matches adamacs_ingest_v2.py logic)
videos_to_process = []

def find_best_video(scan_path, search_str):
    """
    Find the best video file, preferring deinterlaced versions.
    Matches logic from adamacs_ingest_v2.py _ingest_dlc_model()
    """
    # First try to find deinterlaced version
    deinterlaced_videos = list(scan_path.glob(f'*{search_str}*deinterlaced*.mp4'))
    if deinterlaced_videos:
        return deinterlaced_videos[0], True  # Return first match and flag as deinterlaced
    
    # Fallback to regular video
    regular_videos = list(scan_path.glob(f'*{search_str}*.mp4'))
    if regular_videos:
        # Filter out any that might have 'deinterlaced' (shouldn't happen but safety check)
        regular_videos = [v for v in regular_videos if 'deinterlaced' not in v.name]
        if regular_videos:
            return regular_videos[0], False
    
    return None, False

for _, row in tqdm(scans_df.iterrows(), total=len(scans_df), desc="Scanning"):
    scan_path = pathlib.Path(row['path'])
    if not scan_path.exists():
        continue
    
    session_id = row['session_id']
    scan_id = row['scan_id']
    
    # Left eye (eye1 pattern)
    video, has_deinterlaced = find_best_video(scan_path, SEARCH_STR_LEFT)
    if video:
        videos_to_process.append({
            'session_id': session_id,
            'scan_id': scan_id,
            'video_path': str(video),
            'video_name': video.name,
            'eye': 'left',
            'camera': CAMERAS['left'],
            'recording_id': f"{scan_id}_{CAMERAS['left']}",
            'has_deinterlaced': has_deinterlaced
        })
    
    # Right eye (eye2 pattern)
    video, has_deinterlaced = find_best_video(scan_path, SEARCH_STR_RIGHT)
    if video:
        videos_to_process.append({
            'session_id': session_id,
            'scan_id': scan_id,
            'video_path': str(video),
            'video_name': video.name,
            'eye': 'right',
            'camera': CAMERAS['right'],
            'recording_id': f"{scan_id}_{CAMERAS['right']}",
            'has_deinterlaced': has_deinterlaced
        })

videos_df = pd.DataFrame(videos_to_process)
print(f"\nFound {len(videos_df)} eye videos")
print(f"  Left: {len(videos_df[videos_df['eye'] == 'left'])}")
print(f"  Right: {len(videos_df[videos_df['eye'] == 'right'])}")
if len(videos_df) > 0:
    print(f"  Deinterlaced: {len(videos_df[videos_df['has_deinterlaced'] == True])}")
    print(f"  Regular: {len(videos_df[videos_df['has_deinterlaced'] == False])}")

In [ ]:
# Filter out already processed videos
# BUT: Re-process if existing was done on non-deinterlaced video and deinterlaced is now available
to_process = []
to_reprocess = []  # Track videos that need re-ingestion due to deinterlaced availability

for _, row in videos_df.iterrows():
    existing_task = (
        model.PoseEstimationTaskNew 
        & f'recording_id = "{row["recording_id"]}"'
        & f'scan_id = "{row["scan_id"]}"'
        & f'model_name = "{DLC_MODEL_NAME}"'
    )
    
    if not existing_task:
        # No existing task - needs processing
        to_process.append(row.to_dict())
    elif row['has_deinterlaced']:
        # Task exists, but we have deinterlaced video - check if existing used deinterlaced
        existing_file = (
            model.VideoRecordingNew.File 
            & f'recording_id = "{row["recording_id"]}"'
            & f'scan_id = "{row["scan_id"]}"'
        )
        if existing_file:
            existing_path = existing_file.fetch1('file_path')
            if 'deinterlaced' not in existing_path:
                # Existing task used non-deinterlaced video, need to redo
                to_reprocess.append(row.to_dict())
                print(f"  ⚠️  {row['scan_id']} {row['eye']}: Will re-ingest (deinterlaced now available)")

print(f"\nAlready processed (correct video): {len(videos_df) - len(to_process) - len(to_reprocess)}")
print(f"New to process: {len(to_process)}")
print(f"To RE-process (deinterlaced upgrade): {len(to_reprocess)}")
print(f"Total to ingest: {len(to_process) + len(to_reprocess)}")

In [ ]:
def ingest_eye_video(video_info, model_name, use_cropping=False, task_mode='trigger', force_update=False):
    """
    Ingest eye video for DLC processing.
    Based on adamacs_ingest_v2.py _ingest_dlc_model() logic.
    
    Args:
        force_update: If True, update existing file path to new video (for deinterlaced upgrade)
    """
    try:
        # Build key (must include scan_id per VideoRecordingNew schema)
        key = {
            'session_id': video_info['session_id'],
            'scan_id': video_info['scan_id'],
            'recording_id': video_info['recording_id'],
            'camera': video_info['camera']
        }
        
        # Insert video recording
        model.VideoRecordingNew.insert1(key, skip_duplicates=True)
        
        # Insert/update video file
        file_key = {
            'session_id': key['session_id'],
            'scan_id': key['scan_id'],
            'recording_id': key['recording_id'],
            'file_path': video_info['video_path'],
            'file_id': 0
        }
        
        if force_update:
            # For deinterlaced upgrade: delete old file entry and pose estimation entries
            existing_file = (
                model.VideoRecordingNew.File 
                & f'recording_id = "{key["recording_id"]}"'
                & f'scan_id = "{key["scan_id"]}"'
            )
            if existing_file:
                # Delete downstream pose estimation data first (referential integrity)
                (model.PoseEstimationNew & existing_file.fetch1('KEY')).delete()
                (model.PoseEstimationTaskNew & existing_file.fetch1('KEY')).delete()
                existing_file.delete()
            
            # Also delete RecordingInfo if it exists
            existing_info = (
                model.RecordingInfoNew 
                & f'recording_id = "{key["recording_id"]}"'
                & f'scan_id = "{key["scan_id"]}"'
            )
            if existing_info:
                existing_info.delete()
        
        model.VideoRecordingNew.File.insert1(file_key, ignore_extra_fields=True, skip_duplicates=True)
        
        # Create pose estimation task
        task_key = (model.VideoRecordingNew 
                   & f'recording_id="{key["recording_id"]}"' 
                   & f'scan_id="{key["scan_id"]}"').fetch1('KEY')
        
        # Set analyze_videos_params
        if use_cropping:
            params = {'save_as_csv': True, 'dynamic': (True, 0.5, 60)}
        else:
            params = {'save_as_csv': True}
        
        # Pass task_mode explicitly to override auto-detection
        model.PoseEstimationTaskNew.insert_estimation_task(
            task_key, model_name, analyze_videos_params=params, task_mode=task_mode
        )
        return True
        
    except Exception as e:
        print(f"  ❌ {video_info['video_name']}: {e}")
        return False

In [ ]:
# Run batch ingestion
all_to_ingest = to_process + to_reprocess

if not all_to_ingest:
    print("✅ All videos already processed with correct (deinterlaced) files!")
else:
    success, failed = 0, 0
    
    # Process new videos
    if to_process:
        print(f"\n--- Processing {len(to_process)} NEW videos ---")
        for video_info in tqdm(to_process, desc="New videos"):
            # Use DLC_MODEL_NAME if has_deinterlaced (video with 'deinterlaced' tag), 
            # else DLC_DEINT_MODEL_NAME (video is already deinterlaced without tag)
            model_name = DLC_MODEL_NAME if video_info.get('has_deinterlaced', False) else DLC_DEINT_MODEL_NAME
            if ingest_eye_video(video_info, model_name, USE_DYNAMIC_CROPPING, TASK_MODE, force_update=False):
                success += 1
            else:
                failed += 1
    
    # Re-process videos that need deinterlaced upgrade
    if to_reprocess:
        print(f"\n--- RE-PROCESSING {len(to_reprocess)} videos (deinterlaced upgrade) ---")
        for video_info in tqdm(to_reprocess, desc="Deinterlaced upgrade"):
            # Use DLC_MODEL_NAME if has_deinterlaced (video with 'deinterlaced' tag), 
            # else DLC_DEINT_MODEL_NAME (video is already deinterlaced without tag)
            model_name = DLC_MODEL_NAME if video_info.get('has_deinterlaced', False) else DLC_DEINT_MODEL_NAME
            if ingest_eye_video(video_info, model_name, USE_DYNAMIC_CROPPING, TASK_MODE, force_update=True):
                success += 1
            else:
                failed += 1
    
    print(f"\n✅ Successful: {success}")
    print(f"❌ Failed: {failed}")

In [ ]:
# ============================================================================
# FIX: Update any existing 'load' mode tasks to 'trigger' mode
# ============================================================================
# If tasks were previously created with 'load' mode, update them to 'trigger'

# Find tasks with 'load' mode for our models
load_mode_tasks = (
    model.PoseEstimationTaskNew 
    & 'task_mode = "load"'
    & [f'model_name = "{DLC_MODEL_NAME}"', f'model_name = "{DLC_DEINT_MODEL_NAME}"']
)

print(f"Tasks with 'load' mode: {len(load_mode_tasks)}")

if load_mode_tasks:
    print("\nUpdating to 'trigger' mode...")
    
    # Update each task to trigger mode
    for task in tqdm(load_mode_tasks.fetch(as_dict=True), desc="Updating"):
        # Update task_mode to 'trigger'
        (model.PoseEstimationTaskNew & task).delete()
        task['task_mode'] = 'trigger'
        model.PoseEstimationTaskNew.insert1(task, skip_duplicates=True)
    
    print("✅ All tasks updated to trigger mode!")
else:
    print("✅ No 'load' mode tasks found - all good!")

In [ ]:
# Verify results
START_DATE = '2025-05-19'
sessi = 'sess9FTZJR4Y'
query_sessions = (
    session.Session * session.SessionUser * subject.User 
    & f"initials = '{USER_INITIALS}'"
    & f"session_datetime >= '{START_DATE}'"
    # & f'session_id = "{sessi}"'
)

total_tasks = model.PoseEstimationTaskNew & f'model_name = "{DLC_MODEL_NAME}"' & query_sessions
total_tasks_deint = model.PoseEstimationTaskNew & f'model_name = "{DLC_DEINT_MODEL_NAME}"' & query_sessions
completed = model.PoseEstimationNew & f'model_name = "{DLC_MODEL_NAME}"' & query_sessions
completed_deint = model.PoseEstimationNew & f'model_name = "{DLC_DEINT_MODEL_NAME}"' & query_sessions
pending = total_tasks - model.PoseEstimationNew

print(f"Total tasks: {len(total_tasks)}")
print(f"Total tasks deint: {len(total_tasks_deint)}")
print(f"Completed: {len(completed)}")
print(f"Completed deint: {len(completed_deint)}")
print(f"Pending: {len(pending)}")

In [ ]:
completed